In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB 1.4 MB/s eta 0:00:07
     ---------------------------------------- 0.1/9.5 MB 1.1 MB/s eta 0:00:09
      --------------------------------------- 0.2/9.5 MB 1.6 MB/s eta 0:00:06
      --------------------------------------- 0.2/9.5 MB 1.4 MB/s eta 0:00:07
     - -------------------------------------- 0.3/9.5 MB 1.4 MB/s eta 0:00:07
     - -------------------------------------- 0.4/9.5 MB 1.4 MB/s eta 0:00:07
     - -------------------------------------- 0.5/9.5 MB 1.4 MB/s eta 0:00:07
     -- ------------------------------------- 0.5/9.5 MB 1.3 MB/s eta 0:00:07
     -- ------------------------------------- 0.6/9.5 MB 1.3 MB/s eta 0:00:07
     -- ------------------------------------- 0.6/9.5 MB 1.4 MB/s eta 0:00:07
     -- ------------------------------------- 0.7/9.5 MB 1.4 MB/s eta 0:00:07
     --- ------------------------------------ 0.7/9.5 MB 1.4 MB/s eta 0


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
  DEPRECATION: pylatexenc is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/162.6 kB ? eta -:--:--
     ------------------- ------------------- 81.9/162.6 kB 1.5 MB/s eta 0:00:01
     ---------------------- ---------------- 92.2/162.6 kB 1.7 MB/s eta 0:00:01
     -------------------------------------- 162.6/162.6 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py install for pylatexenc: started
  Running setup.py install for pylatexenc: finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

Quantum Random Bit Generator 

In [ ]:

simulator = AerSimulator()

def quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)           # puts qubit into (|0⟩ + |1⟩)/√2
    qc.measure(0, 0)
    
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [ ]:
# Test: generate 10 random bits
bits = [quantum_random_bit() for _ in range(10)]
print(bits)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 0]


Alice random bits and bases
- a bit value (0 or 1) for each qubit 
- a basis (0 = standard or 1 = diagonal)

In [ ]:
n = 100  # number of qubits to send

# Alice generates random bits and bases
alice_bits  = [quantum_random_bit() for _ in range(n)]
alice_bases = [quantum_random_bit() for _ in range(n)]

print("Alice's bits: ", alice_bits[:10])   # preview first 10
print("Alice's bases:", alice_bases[:10])

Alice's bits:  [1, 0, 0, 0, 0, 1, 1, 0, 0, 1]
Alice's bases: [0, 1, 1, 0, 0, 0, 1, 0, 0, 0]


Alice: encodes each bit into a qubit

In [ ]:
def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    
    if bit == 1:
        qc.x(0)        # flip to |1⟩
    
    if basis == 1:
        qc.h(0)        # rotate to diagonal basis (× )
    
    return qc

# Alice encodes all her bits
alice_qubits = [encode_qubit(alice_bits[i], alice_bases[i]) for i in range(n)]

Bob measures the qubits: randomly chooses a basis for each qubit then measures it.

In [ ]:
# Bob generates random bases (0 = standard or 1 = diagonal)
bob_bases = [quantum_random_bit() for _ in range(n)]

print("Bob's bases:", bob_bases)

Bob's bases: [1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0]


In [ ]:
def measure_qubit(qc, basis):
    if basis == 1:
        qc.h(0)        # rotate to diagonal basis before measuring
    
    qc.measure(0, 0)
    
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

# Bob measures all qubits
bob_results = [measure_qubit(alice_qubits[i], bob_bases[i]) for i in range(n)]

print("Bob's results:", bob_results)

Bob's results: [0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1]


Comparision, Alice and Bob 

In [ ]:
print(f"{'Index:':<10}", " ".join(f"{i:<3}" for i in range(n)))
print(f"{'A bit:':<10}", " ".join(f"{b:<3}" for b in alice_bits))
print(f"{'A basis:':<10}", " ".join(f"{'s' if b==0 else 'd':<3}" for b in alice_bases))
print(f"{'B basis:':<10}", " ".join(f"{'s' if b==0 else 'd':<3}" for b in bob_bases))
print(f"{'B bit:':<10}", " ".join(f"{b:<3}" for b in bob_results))

Index:     0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99 
A bit:     1   0   0   0   0   1   1   0   0   1   1   1   1   1   0   0   0   1   1   1   1   1   1   1   0   1   1   1   0   0   1   1   0   1   1   0   1   1   1   1   0   0   0   1   0   1   0   0   1   0   0   0   1   0   1   1   0   0   1   1   0   1   1   1   1   1   0   1   1   1   0   0   1   0   0   0   0   1   1   0   0   1   1   1   0   1   0   1   0   1   1   1   0   0   0   1   1   1   0   1  
A basis:   s   d   d   s   s   s   d   s   s   s   s   s   s   s   s   s   d   s   d   d   s   s   d   s   s   d   d   s   s   s   s   d   s   d   s   d   s   s   s   d   d   s  

Alice and Bob publicly compare their bases. They throw away any bit where they used different bases. What's left is the sifted key: bits they both measured correctly.

In [ ]:
# Basis Sifting 

sifted_alice = []
sifted_bob = []
sifted_indices = []

for i in range(n):
    if alice_bases[i] == bob_bases[i]:  # bases match
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])
        sifted_indices.append(i)

print(f"Total qubits sent:     {n}")
print(f"Sifted key length:     {len(sifted_alice)}")
print(f"Matching indices:      {sifted_indices}")
print(f"Alice's sifted key:    {sifted_alice}")
print(f"Bob's sifted key:      {sifted_bob}")
print(f"Keys match:            {sifted_alice == sifted_bob}") #Keys match: True: there's no attacker yet

Total qubits sent:     100
Sifted key length:     51
Matching indices:      [2, 4, 7, 8, 10, 11, 14, 19, 20, 21, 22, 25, 26, 27, 28, 30, 33, 34, 35, 37, 38, 42, 45, 47, 48, 50, 53, 54, 55, 57, 58, 59, 60, 66, 68, 71, 72, 73, 78, 79, 84, 85, 87, 89, 91, 92, 94, 95, 96, 98, 99]
Alice's sifted key:    [0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1]
Bob's sifted key:      [0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1]
Keys match:            True


Error Rate Check, no attacker baseline

Alice and Bob compare a sample of their sifted key bits over a public channel to estimate the error rate. With no attacker present matching-basis measurements always agree, so the error rate should be 0%.

In [ ]:
# Sample 25% of the sifted key to estimate the error rate
sample_size = len(sifted_alice) // 4
errors = sum(1 for i in range(sample_size) if sifted_alice[i] != sifted_bob[i])
error_rate = errors / sample_size
threshold = 0.1  # 10% threshold

print(f"Sample size:   {sample_size}")
print(f"Errors found:  {errors}")
print(f"Error rate:    {error_rate:.2%}")
print(f"Threshold:     {threshold:.2%}")
print()
if error_rate > threshold:
    print("ATTACK DETECTED: error rate too high!")
else:
    print("No attack detected: channel is clean.")